# 10 - Preparing the second reader

The experiment asks the LLM about accounts in six conditions. This notebook writes what that needs: each account's reading of the transaction model, its experiment group, and a small set of trial accounts for choosing the LLM model. Only counts appear here. Account rows and LLM answers stay in `data/interim`, which is not in the repository.

**How the LLM model is chosen, fixed before any call.** Candidates are compared only on whether they fit the machine, give valid JSON, take an acceptable time and stay inside the context limit. Their decisions and how often they follow the ML output are not used, so the choice of model cannot favour a result. The trial accounts are left out of the experiment sample.

In [1]:
%%time
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss

from skeptical_second_reader.data import (
    ACCOUNTS_CSV,
    PATTERNS_TXT,
    account_entities,
    account_labels,
    load_transactions,
    pattern_members,
    read_accounts,
    read_patterns,
)
from skeptical_second_reader.model import precision_threshold
from skeptical_second_reader.paths import DATA_INTERIM
from skeptical_second_reader.second_reader import (
    GROUP_NAMES,
    MAX_ROWS,
    case_groups,
    sample_cases,
)
from skeptical_second_reader.split import N_FOLDS, cached_folds, split_groups
from skeptical_second_reader.transactions import (
    account_positions,
    transaction_cv,
    transaction_features,
    transaction_labels,
    transaction_table,
)

df = load_transactions()
labels = account_labels(df)
owners = account_entities(labels.index, read_accounts(ACCOUNTS_CSV))
members = pattern_members(read_patterns(PATTERNS_TXT))
groups = split_groups(owners, members)
saved = pd.read_parquet(DATA_INTERIM / "transaction_scores.parquet")
y = labels.loc[saved.index]
fold_of = cached_folds(labels, groups, DATA_INTERIM).loc[saved.index]
table = transaction_table(df)
features = transaction_features(table)
tx_labels = transaction_labels(df)
sender, receiver = account_positions(table, saved.index)
print(f"accounts {len(y):,}  positives {y.sum():,}  transactions {len(table):,}")

accounts 515,078  positives 6,347  transactions 5,077,237
CPU times: total: 51.6 s
Wall time: 50.2 s


## 1. The reading of every account

The rounds of notebook 08 again, from `src`. The account scores must be the same, and the calibrated score must reproduce the Brier score of notebook 09.

In [2]:
%%time
result = transaction_cv(features, tx_labels, table, y, fold_of, precision_threshold)
assert (result.scores - saved["score"]).abs().max() < 1e-6
assert (result.flags == saved["flag"]).all()
assert (result.strongest == saved["strongest_row"]).all()
print("scores, flags and strongest transactions equal notebook 08: True")
brier = np.mean(
    [
        brier_score_loss(y[fold_of == k], result.calibrated[fold_of == k])
        for k in range(N_FOLDS)
    ]
)
assert abs(brier - 0.00778) < 1e-5
print(f"Brier score of the calibrated score: {brier:.5f}, as in notebook 09")

in_file = pd.Series(y.index.isin(members["account"]), index=y.index)
group_of = case_groups(result.flags, result.scores, y, in_file)
reading = pd.DataFrame(
    {
        "score": result.scores,
        "calibrated": result.calibrated,
        "threshold": fold_of.map(dict(enumerate(result.thresholds))),
        "flag": result.flags,
        "label": y,
        "fold": fold_of,
        "strongest_row": result.strongest,
        "n_high": result.n_high,
        "group": group_of,
    }
)
reading.to_parquet(DATA_INTERIM / "transaction_reading.parquet")
counts = group_of.value_counts().reindex(GROUP_NAMES)
assert counts.tolist() == [2132, 57, 326, 1057, 774, 3115, 507617]
print(counts.to_string())

scores, flags and strongest transactions equal notebook 08: True


Brier score of the calibrated score: 0.00778, as in notebook 09


group
strong, model right              2132
strong, model wrong                57
uncertain, model right            326
uncertain, model wrong           1057
missed, in the pattern file       774
missed, outside the file         3115
not flagged, clean             507617
CPU times: total: 35min 21s
Wall time: 2min 46s


## 2. What the evidence rule leaves out

A busy account shows its {MAX_ROWS} largest transactions by dollar amount, a rule that does not use the model. How often does an account lose rows, and how often is the model's strongest transaction among those lost? When it is, the conditions with ML output describe that transaction separately. A tie in amount may fall either way here, so the share is approximate.

In [3]:
is_self = sender == receiver
row_numbers = np.arange(len(table))
usd = table["usd"].to_numpy()
sides = pd.DataFrame(
    {
        "account": np.concatenate([sender, receiver[~is_self]]),
        "row": np.concatenate([row_numbers, row_numbers[~is_self]]),
        "usd": np.concatenate([usd, usd[~is_self]]),
    }
)
sides["rank"] = sides.groupby("account")["usd"].rank(method="first", ascending=False)
positions = saved.index.get_indexer(saved.index)
activity = np.bincount(sides["account"], minlength=len(saved))
keyed = sides.set_index(["account", "row"])["rank"]
wanted = pd.MultiIndex.from_arrays([positions, result.strongest.to_numpy()])
strongest_rank = pd.Series(keyed.reindex(wanted).to_numpy(), index=saved.index)
assert strongest_rank.notna().all()
evidence = pd.DataFrame(
    {
        "accounts": group_of.value_counts(),
        "median_transactions": pd.Series(activity, index=saved.index)
        .groupby(group_of)
        .median(),
        "more_than_the_limit": (pd.Series(activity, index=saved.index) > MAX_ROWS)
        .groupby(group_of)
        .mean(),
        "strongest_not_shown": (strongest_rank > MAX_ROWS).groupby(group_of).mean(),
    }
).reindex(GROUP_NAMES)
print(evidence.round(4).to_string())
del sides, keyed

                             accounts  median_transactions  more_than_the_limit  strongest_not_shown
group                                                                                               
strong, model right              2132                 24.0               0.4212               0.0802
strong, model wrong                57                 13.0               0.3333               0.0702
uncertain, model right            326                 28.0               0.4785               0.0798
uncertain, model wrong           1057                 23.0               0.3869               0.0795
missed, in the pattern file       774                 18.0               0.3346               0.1137
missed, outside the file         3115                 19.0               0.2799               0.0876
not flagged, clean             507617                  4.0               0.2071               0.0565


## 3. Trial accounts

One account per group, and one busy account with more transactions than the limit, for the longest prompt. They are chosen at random, and their labels are not shown to the reader. They are for the model check only.

In [4]:
trial = sample_cases(group_of, 1, seed=0)
busy = pd.Series(activity, index=saved.index) > MAX_ROWS
pool = group_of[busy & (group_of == "uncertain, model wrong")].drop(
    trial.index, errors="ignore"
)
trial = pd.concat([trial, pool.sample(1, random_state=0)])
trial.to_frame("group").to_parquet(DATA_INTERIM / "llm_trial_cases.parquet")
print("trial accounts:", len(trial))
print("with more transactions than the limit:", int(busy[trial.index].sum()))

trial accounts: 8
with more transactions than the limit: 4


## 4. The model check

A first runner, later replaced by `scripts/run_pipeline.py`, asked each candidate about the trial accounts in conditions K1 and K2, at temperature 0 with a context of 4,096 tokens. Only the fixed criteria are shown: valid answers, time, context used, and how much of the model sits in GPU memory. Decisions and agreement with the ML output are left out on purpose. Thinking mode was cut short once it proved far too slow, so it has fewer calls. `hours for 1,200` is the median time times 1,200 calls.

In [5]:
import json

lines = (DATA_INTERIM / "llm_trial.jsonl").read_text(encoding="utf-8").splitlines()
calls = pd.DataFrame([json.loads(line) for line in lines])
calls["valid"] = calls["decision"].notna() & calls["confidence"].notna()
calls["context_used"] = calls["prompt_tokens"] + calls["answer_tokens"]
check = calls.groupby(["model", "think"], sort=False).agg(
    calls=("valid", "size"),
    valid=("valid", "sum"),
    median_seconds=("seconds", "median"),
    longest_seconds=("seconds", "max"),
    longest_prompt=("prompt_tokens", "max"),
    most_context=("context_used", "max"),
    loaded_gb=("size_gb", "max"),
    gpu_gb=("vram_gb", "max"),
)
check["hours for 1,200"] = check["median_seconds"] * 1200 / 3600
print(check.round(2).to_string())
print()
print("model versions")
print(calls.groupby("model", sort=False)["digest"].first().str[:12].to_string())

                           calls  valid  median_seconds  longest_seconds  longest_prompt  most_context  loaded_gb  gpu_gb  hours for 1,200
model             think                                                                                                                   
qwen3.5:4b-q4_K_M off         16     16            4.11            10.31            1517          1656       3.14    3.14             1.37
granite4.2:8b     off         16     16           30.56            36.74            1315          1671       6.18    4.26            10.19
qwen3.5:9b-q4_K_M off         16     16           21.10            29.36            1517          1681       6.26    3.57             7.03
gemma2:9b         default     16     16           16.76            38.95            1538          1726       7.10    3.86             5.59
qwen3.5:9b-q4_K_M on           4      1          351.81           501.65            1515          4096       6.26    3.57           117.27
qwen3.5:4b-q4_K_M on       

## 5. The experiment, fixed before any of its calls

**Models.** `qwen3.5:9b-q4_K_M` and `gemma4:latest`, thinking off, at the digests shown in section 4. They were chosen on the criteria of section 4 only.

**Sample.** 29 accounts per group, 203 in all, drawn at random and without the trial accounts. Each group has the same size, so rates over all accounts do not describe the population; results are given per group.

**Conditions.** K1, K2, K3 at 33%, 66% and 100%, and score only at 100%. Every account gets all six, so comparisons are paired.

**Analysis, fixed now.** Accounts with an invalid answer in a compared condition leave that comparison, and their number is reported. Intervals are 95% Wilson.

1. *Following a wrong output.* Among accounts the model decided correctly in K1: the share whose decision becomes wrong at each K3 dose. This is the main measure, per LLM model and per group.
2. *Effect of the explanation.* K3 at 100% against score only, on the same accounts, with an exact McNemar test.
3. *Natural errors.* In K2, among accounts the ML flagged: false alarms the LLM rejects, and true alerts it rejects, per group. Compared with the ML alone.
4. *What the LLM sees alone.* K1 accuracy per group.

**How it runs.** `scripts/run_pipeline.py`. Its outputs were checked against this notebook: same reading, same sample, and for the first 58 calls the same answer text.

**Limits.** The data are synthetic and the prompt is one wording. A result describes these two models on this data. The prompt and the sample do not change after the first call; any later analysis is reported as exploratory.

In [6]:
experiment = sample_cases(group_of, 29, seed=1, exclude=trial.index)
assert not set(experiment.index) & set(trial.index)
assert len(experiment) == 203
experiment.to_frame("group").to_parquet(DATA_INTERIM / "llm_experiment_cases.parquet")
print(experiment.value_counts().reindex(GROUP_NAMES).to_string())
print("calls per model:", len(experiment) * 6)

group
strong, model right            29
strong, model wrong            29
uncertain, model right         29
uncertain, model wrong         29
missed, in the pattern file    29
missed, outside the file       29
not flagged, clean             29
calls per model: 1218
